# AI-Lake + BigQuery

This notebook targets the **BigQuery emulator** (`goccy/bigquery-emulator`),
which provides the full BigQuery REST API locally — no GCP account required.

Because BigQuery does not natively read Parquet files from a volume, the
notebook loads the fixture via PyArrow, converts rows to JSON, and uses
**streaming inserts** to populate the emulator — the same approach used in
the `compat-heavy.yml` CI workflow.

**Prerequisites:** start the engines compose overlay:
```bash
docker compose \
  -f tests/docker/compose-demo.yml \
  --profile engines \
  up -d
```

**What this notebook covers:**
1. Read AI-Lake Parquet fixture with PyArrow (proves standard Parquet compat)
2. Create BQ dataset + table in the emulator
3. Stream-insert rows (batched)
4. SQL queries via BigQuery client
5. Schema, row count, and field validation

In [ ]:
import base64
import os
import pathlib
import socket

import pyarrow as pa
import pyarrow.parquet as pq

TABLE_PATH = os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')
BQ_HOST    = os.environ.get('BQ_EMULATOR_HOST', 'bigquery-emulator')
BQ_PORT    = int(os.environ.get('BQ_EMULATOR_PORT', '9050'))
BQ_PROJECT = os.environ.get('BQ_PROJECT', 'ailake-demo')
DATASET    = 'ailake_compat'
TABLE      = 'demo_table'

socket.setdefaulttimeout(30)

print(f'BigQuery emulator : {BQ_HOST}:{BQ_PORT}')
print(f'Project           : {BQ_PROJECT}')
print(f'Table             : {BQ_PROJECT}.{DATASET}.{TABLE}')

In [ ]:
# ── Pre-flight check — verify BigQuery emulator is reachable ─────────────────
import socket, urllib.request

BQ_HOST = os.environ.get('BQ_EMULATOR_HOST', 'bigquery-emulator')
BQ_PORT = int(os.environ.get('BQ_EMULATOR_PORT', '9050'))

try:
    with socket.create_connection((BQ_HOST, BQ_PORT), timeout=5):
        print(f"OK: BigQuery emulator reachable at {BQ_HOST}:{BQ_PORT}")
except OSError:
    raise RuntimeError(
        f"\n\nBigQuery emulator is not reachable at {BQ_HOST}:{BQ_PORT}.\n"
        "Start the engines overlay first:\n\n"
        "  docker compose \\\n"
        "    -f tests/docker/compose-demo.yml \\\n"
        "    -f tests/docker/compose-demo-engines.yml \\\n"
        "    up -d\n"
    )

## 1. Read AI-Lake fixture with PyArrow

Demonstrates that standard Parquet readers handle the AI-Lake file format
without errors — the HNSW footer bytes are invisible.

In [ ]:
parquet_files = sorted(pathlib.Path(TABLE_PATH).rglob('*.parquet'))
parts = [pq.read_table(str(p)) for p in parquet_files]
arrow_table = pa.concat_tables(parts)

print(f'Rows read by PyArrow : {len(arrow_table)}')
print(f'Schema               : {arrow_table.schema}')

## 2. Connect to BigQuery emulator

In [ ]:
# Set the emulator host BEFORE importing the BQ client library.
os.environ['BIGQUERY_EMULATOR_HOST'] = f'{BQ_HOST}:{BQ_PORT}'

from google.api_core.client_options import ClientOptions
from google.auth.credentials import AnonymousCredentials
from google.cloud import bigquery

bq = bigquery.Client(
    project=BQ_PROJECT,
    credentials=AnonymousCredentials(),
    client_options=ClientOptions(
        api_endpoint=f'http://{BQ_HOST}:{BQ_PORT}'
    ),
)
print('BigQuery client connected to emulator.')

## 3. Create dataset and table

In [ ]:
bq.create_dataset(DATASET, exists_ok=True)
print(f'Dataset `{DATASET}` ready.')

schema_bq = [
    bigquery.SchemaField('text',      'STRING'),
    bigquery.SchemaField('embedding', 'BYTES'),   # F16 bytes, base64-encoded for JSON
]
table_ref = bq.dataset(DATASET).table(TABLE)
bq.create_table(bigquery.Table(table_ref, schema=schema_bq), exists_ok=True)
print(f'Table `{TABLE}` ready.')

## 4. Stream-insert rows (batched)

In [ ]:
text_col = arrow_table.column('text').to_pylist()
emb_col  = arrow_table.column('embedding').to_pylist()

rows = [
    {
        'text':      str(text_col[i]),
        'embedding': base64.b64encode(bytes(emb_col[i])).decode(),
    }
    for i in range(len(arrow_table))
]

BATCH_SIZE = 500
table_obj = bq.get_table(table_ref)

for start in range(0, len(rows), BATCH_SIZE):
    batch  = rows[start : start + BATCH_SIZE]
    errors = bq.insert_rows_json(table_obj, batch)
    if errors:
        raise RuntimeError(f'Insert errors at offset {start}: {errors}')
    print(f'  inserted rows {start}–{start + len(batch) - 1}')

print(f'\nAll {len(rows)} rows inserted.')

## 5. SQL queries

In [ ]:
count = next(
    bq.query(f'SELECT COUNT(*) FROM `{BQ_PROJECT}.{DATASET}.{TABLE}`').result(timeout=30)
)[0]
print(f'Row count: {count}')
assert count == len(arrow_table), f'Expected {len(arrow_table)}, got {count}'

In [ ]:
import pandas as pd

rows_found = list(
    bq.query(f"""
        SELECT text
        FROM `{BQ_PROJECT}.{DATASET}.{TABLE}`
        WHERE text LIKE '%vector search%'
        LIMIT 5
    """).result(timeout=30)
)
df = pd.DataFrame([dict(r) for r in rows_found])
print(f"Rows about 'vector search': {len(rows_found)}")
df

In [ ]:
agg = list(
    bq.query(f"""
        SELECT
            COUNT(*)             AS total_rows,
            AVG(LENGTH(text))    AS avg_text_chars,
            AVG(LENGTH(embedding)) AS avg_emb_base64_chars
        FROM `{BQ_PROJECT}.{DATASET}.{TABLE}`
    """).result(timeout=30)
)
pd.DataFrame([dict(r) for r in agg])

## 6. Schema inspection

In [ ]:
bq_table = bq.get_table(table_ref)
schema_df = pd.DataFrame(
    [{'field': f.name, 'type': f.field_type, 'mode': f.mode} for f in bq_table.schema]
)
print(f'Rows in table: {bq_table.num_rows}')
schema_df

## Key takeaway

BigQuery compatibility works at **two levels**:

1. **PyArrow reads the AI-Lake Parquet file** without errors — the HNSW section
   after the final `PAR1` bytes is invisible to standard Parquet readers.

2. **Tabular data loads cleanly into BigQuery** — `text` and `embedding` columns
   are valid BigQuery types (`STRING` and `BYTES`). BigQuery treats the embedding
   as opaque bytes; vector search must be performed by the AI-Lake SDK before
   loading results into BigQuery.